## Time with python

En esta sesion se aprendera a usar los modulos time de python para poder calcular el tiempo en ejecucion de funciones.

In [48]:
# Importamos los modulos necesarios y definimos nuestra hora local
import time, calendar, os
os.environ['TZ'] = "America/Bogota"
time.tzset()

In [49]:
print(time.gmtime())
print(time.localtime())

v = time.gmtime()
# Tiempo en segundos de hora UTC
v_c_utc = (calendar.timegm(tuple=v))

v_l = time.localtime()
# Tiempo en segundos de HORA segun mi local machine.
v_c_local = (calendar.timegm(tuple=v_l))

#Como siempre reporta segundos, hacemos una funcion para pasar todo a horas.
def format_hour_posiive(value):
    return round(abs((value/60)/60))

print(f"tiempo en segundos desde epoch con UTC {format_hour_posiive(v_c_utc)}")
print(f"tiempo en segundos desde epoch con Machile Local time {format_hour_posiive(v_c_local)}")
v_h_s = format_hour_posiive((v_c_utc - v_c_local))
print(f"Diferencia entre ambos (horas) {v_h_s}")


time.struct_time(tm_year=2026, tm_mon=7, tm_mday=5, tm_hour=17, tm_min=24, tm_sec=3, tm_wday=6, tm_yday=186, tm_isdst=0)
time.struct_time(tm_year=2026, tm_mon=7, tm_mday=5, tm_hour=12, tm_min=24, tm_sec=3, tm_wday=6, tm_yday=186, tm_isdst=0)
tiempo en segundos desde epoch con UTC 495353
tiempo en segundos desde epoch con Machile Local time 495348
Diferencia entre ambos (horas) 5


Como se observa, siempre habra diferencia entre gmtime() y localtime(), por que una mide en UTC y la otra en la zona local, en ese caso, pusimos nuestra zona en America/Bogota.

Esto se aprendio mas por que venia en la documentaicon y me parecio importante, pero viendolo mejor, la forma de contar segundos, existe una funcion especilizada en eso. time.time(), asi que usar en este caso gmtime() y localtime(), no es necesario.

In [ ]:
# Funcion time.time()

t1 = time.time()
print(t1)

#Esto no el numero en segundos desde la ultima epoca, si imprimimos el utc de gmtime() y este, deberian ser el mismo creeria yo
print("Comprobemos")
print(v_c_utc)
print(t1)
print(f"diferencia {v_c_utc} - {t1} = {v_c_utc-t1}")

#Pues no es el mismo, por que?
#Si es el mismo, pero entre ejecucion esos milisegundos que se pierden los denota la funcion y los envia, al ahcerlo en diferencias lineas de ejecion y como la llamada es secuencial, eso pasa, en general es el mismo.

print(f"Veamoslo asi en la misma linea {time.time()} y {calendar.timegm(time.gmtime())}")

#Ves, casi nada de diferencia :)


1783273292.429371
Comprobemos
1783272243
1783273292.429371
diferencia 1783272243 - 1783273292.429371 = -1049.4293711185455
Veamoslo asi en la misma linea 1783273292.4297516 y 1783273292


Ahora si probemos esto de forma en medicion de tiempo de una funcion, haremos una funcion que tarde en ejecutarse 3 segundos, usando la funcion sleep de time.

In [99]:
#Primero, que es time.sleep?
#Es una funcion que dice, espera el valor en mis parametros x valor antes de seguir con lo siguiente

def time_count_secounds(initial, final):
    return (final - initial)*1000

#Ahora, ejecutaremos time con t1, luego t2, pero en medio pondremos un sleep, haber cuantos segundos no cuenta, usando la funcion de time_count_secounds

t1 = time.time()

time.sleep(1)

t2 = time.time()
v_t = time_count_secounds(t1,t2)
print(f"Tiempo tardado en ejecutarse time.time {v_t}")

# Bien eso nos ayuda, pero usar time.time no es la forma ideal, dentro de la documentacion existe time.perf_counter(), con valores mas precisos, veamos su diferencia.

t1_p = time.perf_counter()

time.sleep(1)

t2_p = time.perf_counter()

v_p = time_count_secounds(t1_p, t2_p)
print(f"Tiempo tardado en ejecutarse time.perf_counter {v_p}")

print(f"diferencia entre ambos {v_p} - {v_t} = {abs((v_p - v_t))}")



Tiempo tardado en ejecutarse time.time 1000.4281997680664
Tiempo tardado en ejecutarse time.perf_counter 1000.4977109992979
diferencia entre ambos 1000.4977109992979 - 1000.4281997680664 = 0.06951123123144498


El resultado es casi el mismo, con milisegundos pero time.perf_counter es mucho ams exacto en el tiempo tardado, pueden ser milisegundos, nano o micro, pero en un diseño donde necesitamos saber cuanto tardo una funciona que muchas se hacen en nada, es muy importante tener certeza del resultado, siempre usar time.perf_counter antes de time.time(), time.time() esta mas pensando para medir tiempo en horas, tiempo en el relog, no en medir intervalos de tiempo, como lo si esta time.perf_counter.

In [138]:
#Ahora lo probaremos en dos funciones reales, 
# - una funcion que dado un rango, suma todos los valores del rango usando range
# - una funcion que multiplique varias veces un valor x un numero random y de el resultado.

#Primero entendamos que hace la funcion range
#recibe 3 parametros, uno inicial y otro final, y otro que son los pasos.
#comunmente usado en cliclos para iterar sobre rangos, incluso se pueden crear listas, digamos, queremos una lista de 10 numero, que encima solo cuente numeros de 2 en 2, podemos hacer algo tipo
print(list(range(2, 20, 2)))
#Por que ponemos 20 de valor final, si queremos solo 10? por que si saltamos de dos en 2 y ponemos 10, nos dara solo una lista de 5 numeros, por que no el parametro 2, no es numero de la lista, es el valor hasta donde ira nuestra lista, es decir donde parara de hacer su magia.


#Prmero entendamos ramdon modulo :)
import random
print(random.uniform(1,10))
#Genera un numero "aleatorio", podemos usarlo para generar una lista aleatoria, como mira
list_random = []

for i in range(10):
    list_random.append(random.uniform(1,10))
    
print(list_random)
#Esmuy util y lo usaremos para nuestra funcion:)

#haremos una funcion, que sume los valores de esta lista y usaremos lo de antes para saber cuanto tarda en ejecutarse.



def sum_values_to_range():
    #Empezamos con el primer contador
    t1 = time.perf_counter()
    #Ejecutamos lo que queremos
    sum_range = sum(range(2, 20, 2))
    #final time
    t2 = time.perf_counter()
    print(f"Suma que queriamos {sum_range}, tiempo que tardo {time_count_secounds(t1,t2)}")
    
def mutlipe_values_to_random():
    #Inicializamos el tiempo
    t1 = time.perf_counter()
    #Creamos una lista, misma que antes
    list_multiple = list(range(2,20,2))
    
    for i in range(len(list_multiple)):
        #Vamos por cada valor y multiplicamos cada valor por un numero random
        value_multpli = list_multiple[i] * random.uniform(1,10)
        list_multiple[i] = value_multpli
    #Ahora, multipliqueramos toda la lista usando el modulo math
    import math
    # Usando pro, podemos sacar el producto dada una lista :)
    multi_range = math.prod(list_multiple)        
    #Fianlizamos el tiempo
    t2 = time.perf_counter()
    print(f"Multiplcacion random que queriamos {multi_range}, tiempo que tardo {time_count_secounds(t1, t2)}")
        
        
    
sum_values_to_range()
mutlipe_values_to_random()



[2, 4, 6, 8, 10, 12, 14, 16, 18]
9.82256290568574
[4.410841880870824, 1.5575006486978835, 4.397318934959127, 8.251328884823536, 8.459005712923428, 7.092495379161841, 6.045990236939665, 8.141999835044153, 6.916624748215548, 7.217936306555788]
Suma que queriamos 90, tiempo que tardo 0.002085000232909806
Multiplcacion random que queriamos 126887534675935.97, tiempo que tardo 0.01036100002238527


Ahora timeit, que es buenoa la forma realmente que nos provee python para medir el tiempo de una funcion de forma mas exacta, ya que esta tiene en cuenta el tiempo de la CPU, y desactivando el recolector de basura para tener una medicion mas exacta.

In [152]:
import timeit, math
# Esto nos ayuda a medir la funcion con un valor promedio dado una ronda especifica de mediciones que se hace y un numero de veces :)

#Primero empecemos con repeat, que nos ayudaa saber dado un numero de repeticiones que nos pasara en una lista y el number, que es el numeor de veces que iterara y stmt, es la funcion que medita, usemos la funcion de antes sum_values_to_range y mutlipe_values_to_random, pero sin tiempos dentro

def sum_values_to_range_with_times():
    return sum(range(2, 20, 2))

def mutlipe_values_to_random_with_times():
    list_multiple = list(range(2,20,2))
    
    for i in range(len(list_multiple)):
        #Vamos por cada valor y multiplicamos cada valor por un numero random
        value_multpli = list_multiple[i] * random.uniform(1,10)
        list_multiple[i] = value_multpli
    #Ahora, multipliqueramos toda la lista usando el modulo math
    # Usando pro, podemos sacar el producto dada una lista :)
    return math.prod(list_multiple)  
        
tiempos_1 = timeit.repeat(stmt=sum_values_to_range_with_times, repeat=5, number=100000)
tiempos_2 =timeit.repeat(stmt=mutlipe_values_to_random_with_times, repeat=5, number=100000)

print(f"Tiempos de la funcion 1 {tiempos_1}")
print(f"Tiempos de la funcion 2 {tiempos_2}")

#Para saber cuanto tardo realmente, usamo el valor minimo de ambas listas, por que el valor menor? simple, siempre que medimos ese valor necesitamos saber cual fue el mejor, por que asi quitamos problemas que quizas uso con la CPU demas, asi que si, el valor minimo es el mejor valor para definir cuanto tarda la funcion.

print(f"Minimo real, es el que define cuanto demoro, de la primera {min(tiempos_1)}")
print(f"Minimo real, es el que define cuanto demoro, de la primera {min(tiempos_2)}")



Tiempos de la funcion 1 [0.027565664000576362, 0.029397455999060185, 0.026418775001729955, 0.0263537440005166, 0.026348822999352706]
Tiempos de la funcion 2 [0.3083597179993376, 0.2873974360009015, 0.2858861410004465, 0.292958947000443, 0.3059878000003664]
Minimo real, es el que define cuanto demoro, de la primera 0.026348822999352706
Minimo real, es el que define cuanto demoro, de la primera 0.2858861410004465


In [154]:
#Refactorizando la funcion for de multiplicar para tenerlo en cuenta, usamos enumerate()
#Esta funcion nos permite vincular en el for, directamente el valor del index i de la lista a un variable, en este caso llamada x 

def refactor_mutlipe_values_to_random_with_times():
    list_random = list(range(2, 20, 2))
    for i, x in enumerate(list_random):
        list_random[i] = x * random.uniform(1,10)
        
    return math.prod(list_random)

tiempos_2_refactor = timeit.repeat(stmt=refactor_mutlipe_values_to_random_with_times, repeat=5, number=100000)

def super_refactor_mutlipe_values_to_random_with_times():
    list_multiple = list(range(2, 20, 2))
    list_multiple = [x * random.uniform(1,10) for x in list_multiple]
    return math.prod(list_multiple)

tiempos_3_refactor = timeit.repeat(stmt=super_refactor_mutlipe_values_to_random_with_times, repeat=5, number=100000)


print(f"Tiempo funcion 2 refactorizada {tiempos_2_refactor}")
print(f"Tiermpo tardado minimo {min(tiempos_2_refactor)}")
print(f"Tiempos funciton 2 super refactor {tiempos_3_refactor}")
print(f"Tiermpo tardado minimo {min(tiempos_3_refactor)}")


Tiempo funcion 2 refactorizada [0.2940486059997056, 0.3129582340006891, 0.2890604780004651, 0.289510810000138, 0.3050270970015845]
Tiermpo tardado minimo 0.2890604780004651
Tiempos funciton 2 super refactor [0.2843994319991907, 0.2792777049999131, 0.5173966439997457, 0.5269788500008872, 0.572540179000498]
Tiermpo tardado minimo 0.2792777049999131
